# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [4]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
MODEL_GEMINI = 'gemini-2.5-flash'
MODEL_GEMMA = 'gemma3:1b'

In [12]:
# constants
system_prompt = ''' you are a code expert who answers the technical questions in simple terms with example.
example: question asked = "why are arrays indexed as 0 in c++ ?", answer = "the first item of the array is same as array address, this comes handy in calculating further items address as the cpu has just do one operation to calculate the address i.e. for ith item address will be *(arr + i * 4) considering it's int type array."
'''

In [13]:
# set up environment
from IPython.display import Markdown, display, update_display
def stream_answers(question):
    stream = client.chat.completions.create(
        model=MODEL_GEMINI,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [14]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [15]:
# Get gemini to answer, with streaming
stream_answers(question)

Let's break down this line of Python code piece by piece, like looking at different parts of a machine, to understand what it does and why it's built that way.

Imagine you have a big pile of book records, and each record is like a little dictionary with information about one book (title, author, etc.).

---

### What it does:

This code **finds all the unique authors** from a collection of `books`, *ignoring* any books that don't have an author specified. Then, it **"yields" each of these unique authors one by one**.

---

### Why it does it:

Let's dissect the line: `yield from {book.get("author") for book in books if book.get("author")}`

1.  **`for book in books`**:
    *   **What it does:** This is the first step of a loop. It tells Python to go through each individual `book` (which we assume is a dictionary) in the larger collection called `books` (which is likely a list of these book dictionaries).
    *   **Why:** You want to look at every single book record to find all the authors.

2.  **`if book.get("author")`**:
    *   **What it does:** This is a filter.
        *   `book.get("author")`: This safely tries to get the value associated with the key `"author"` from the current `book` dictionary. If a `book` doesn't have an "author" key, it gracefully returns `None` instead of causing an error.
        *   The `if` part then checks if what `book.get("author")` returned is "truthy". In Python, an empty string (`""`) or `None` are considered "falsy", while actual names (like `"Stephen King"`) are "truthy".
    *   **Why:** You only care about books that *actually have* an author name. You want to skip books where the author field is missing or empty. This makes the code robust against incomplete data.

3.  **`book.get("author")` (the first one)**:
    *   **What it does:** For every book that passes the `if` test (meaning it has a valid author), this part extracts that author's name.
    *   **Why:** This is the specific piece of information (the author's name) you want to collect.

4.  **`{ ... }` (the curly braces around the whole comprehension)**:
    *   **What it does:** This creates a **set**. A set is a collection that automatically ensures all its items are unique. If you try to add the same item twice, it only stores it once.
    *   **Why:** This is the clever part for getting *unique* authors. If "J.K. Rowling" wrote 7 books in your collection, her name will only appear once in the resulting set.

    *   **So, up to this point, `{book.get("author") for book in books if book.get("author")}` creates a set of all unique author names from your books, skipping empty or missing author entries.**

5.  **`yield from ...`**:
    *   **What it does:** This is a Python feature used inside a **generator function**. It takes an "iterable" (like our set of unique authors) and effectively "yields" each item from that iterable one by one. It's a shortcut for writing:
        ```python
        for item in my_set_of_authors:
            yield item
        ```
    *   **Why:** When you use `yield from`, you're telling your generator function, "Just pass along all the items from this other collection." It makes your code cleaner and more efficient if you're working with generators, as it avoids loading all items into memory at once if the iterable was very large (though for a set, it does create the whole set first). It allows the calling code to ask for authors one at a time, rather than getting a full list of all unique authors upfront.

---

### In Simple Terms with an Example:

Imagine you have these books:

```python
books = [
    {"title": "Book A", "author": "Alice"},
    {"title": "Book B", "author": "Bob"},
    {"title": "Book C", "author": "Alice"}, # Alice again
    {"title": "Book D"},                   # No author
    {"title": "Book E", "author": ""},     # Empty author
    {"title": "Book F", "author": "Charlie"}
]
```

1.  **`{book.get("author") for book in books if book.get("author")}`** would do this:
    *   "Alice" from "Book A" (Valid) -> Add "Alice"
    *   "Bob" from "Book B" (Valid) -> Add "Bob"
    *   "Alice" from "Book C" (Valid) -> Add "Alice" (Set ignores duplicate, still just one "Alice")
    *   `None` from "Book D" (Falsy) -> Skip
    *   `""` from "Book E" (Falsy) -> Skip
    *   "Charlie" from "Book F" (Valid) -> Add "Charlie"

    The result of this comprehension would be a set like `{"Alice", "Bob", "Charlie"}` (the order might vary because sets don't guarantee order).

2.  **`yield from {"Alice", "Bob", "Charlie"}`** would then:
    *   `yield "Alice"`
    *   `yield "Bob"`
    *   `yield "Charlie"`

So, if this line was inside a generator function, calling that function would give you back these three unique author names, one after the other, as you requested them.